# UFC Stats Scraper

| Cell | Stage | Output |
|------|-------|--------|
| 1 | Setup | session, helpers |
| 2 | Event list | `data/events.csv` |
| 3 | Event fights | `data/fights_raw.csv` |
| 4 | Fighter directory | `data/fighters.csv` |
| 5 | Fighter profiles | `data/fighters_full.csv` |
| 6 | Fight details | `data/fight_details.json` |
| 7 | Rebuild corners | `data/fights.csv` |
| 8 | Summary | verification |

**F1 = Red corner = Favorite | F2 = Blue corner = Underdog**

In [1]:
# notebooks/01_scraper.ipynb — Cell 1: Imports & Setup

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from tqdm import tqdm
import os
import json
import concurrent.futures

BASE_URL = 'http://www.ufcstats.com'
HEADERS = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36'}
WORKERS = 10
DATA_DIR = './data'
os.makedirs(DATA_DIR, exist_ok=True)

session = requests.Session()
session.headers.update(HEADERS)

def get_soup(url):
    resp = session.get(url)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, 'lxml')

print(f'Ready — {WORKERS} threads')

Ready — 10 threads


In [2]:
# notebooks/01_scraper.ipynb — Cell 2: Scrape Event List
# Date is inside TD[0] alongside event name. TD[1] is location.
# Output: data/events.csv

def scrape_all_events():
    url = f'{BASE_URL}/statistics/events/completed?page=all'
    soup = get_soup(url)
    events = []
    for row in soup.select('tr.b-statistics__table-row'):
        link = row.select_one('a.b-link')
        if not link:
            continue
        cells = row.select('td')
        td0 = cells[0].get_text(separator='|', strip=True).split('|')
        events.append({
            'event_name': td0[0].strip(),
            'event_url': link['href'],
            'event_date': td0[1].strip() if len(td0) > 1 else '',
            'location': cells[1].text.strip() if len(cells) > 1 else ''
        })
    return events

events = scrape_all_events()
events_df = pd.DataFrame(events)
events_df.to_csv(f'{DATA_DIR}/events.csv', index=False)
print(f'Saved {len(events_df)} events to data/events.csv')
events_df[['event_name','event_date','location']].head()

Saved 769 events to data/events.csv


,event_name,event_date,location
0,UFC 327: Prochazka vs. Ulberg,"April 11, 2026","Miami, Florida, USA"
1,UFC Fight Night: Moicano vs. Duncan,"April 04, 2026","Las Vegas, Nevada, USA"
2,UFC Fight Night: Adesanya vs. Pyfer,"March 28, 2026","Seattle, Washington, USA"
3,UFC Fight Night: Evloev vs. Murphy,"March 21, 2026","London, England, United Kingdom"
4,UFC Fight Night: Emmett vs. Vallejos,"March 14, 2026","Las Vegas, Nevada, USA"


In [3]:
# notebooks/01_scraper.ipynb — Cell 3: Scrape Event Fights
# Event page always lists WINNER first — NOT red corner.
# We fix corner order in Cell 7 using fight detail pages.
# Output: data/fights_raw.csv

def scrape_event_fights(event_row):
    try:
        soup = get_soup(event_row['event_url'])
    except:
        return []
    fights = []
    for row in soup.select('tr.b-fight-details__table-row')[1:]:
        cols = row.select('td')
        if len(cols) < 10:
            continue
        links = cols[1].select('a')
        if len(links) < 2:
            continue
        flag = row.select_one('a.b-flag')
        gv = lambda c: [p.text.strip() for p in c.select('p')]
        kd, st, td, sb = gv(cols[2]), gv(cols[3]), gv(cols[4]), gv(cols[5])
        fights.append({
            'event_name': event_row['event_name'],
            'event_date': event_row['event_date'],
            'fight_url': flag['href'] if flag else None,
            'fighter_1_winner': links[0].text.strip(),
            'fighter_2_loser': links[1].text.strip(),
            'winner_kd': kd[0] if kd else None,
            'loser_kd': kd[1] if len(kd)>1 else None,
            'winner_str': st[0] if st else None,
            'loser_str': st[1] if len(st)>1 else None,
            'winner_td': td[0] if td else None,
            'loser_td': td[1] if len(td)>1 else None,
            'winner_sub': sb[0] if sb else None,
            'loser_sub': sb[1] if len(sb)>1 else None,
            'weight_class': cols[6].text.strip(),
            'method': cols[7].text.strip(),
            'round': cols[8].text.strip(),
            'time': cols[9].text.strip(),
        })
    return fights

all_fights = []
print(f'Scraping fights from {len(events_df)} events...')
with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = {ex.submit(scrape_event_fights, r): r
            for r in events_df.to_dict('records')}
    for fut in tqdm(concurrent.futures.as_completed(futs),
                    total=len(futs), desc='Events'):
        res = fut.result()
        if res:
            all_fights.extend(res)

fights_raw_df = pd.DataFrame(all_fights)
fights_raw_df.to_csv(f'{DATA_DIR}/fights_raw.csv', index=False)
print(f'\nSaved {len(fights_raw_df)} fights to data/fights_raw.csv')

Scraping fights from 769 events...


Events: 100%|██████████| 769/769 [00:53<00:00, 14.45it/s]



Saved 8637 fights to data/fights_raw.csv


In [4]:
# notebooks/01_scraper.ipynb — Cell 4: Scrape Fighter Directory
# Gets basic info + fighter_url. We FOLLOW URLs in Cell 5.
# Output: data/fighters.csv

def scrape_fighters_for_letter(char):
    try:
        soup = get_soup(f'{BASE_URL}/statistics/fighters?char={char}&page=all')
    except:
        return []
    out = []
    for row in soup.select('tr.b-statistics__table-row'):
        cols = row.select('td')
        if len(cols) < 10:
            continue
        link = cols[0].select_one('a')
        if not link:
            continue
        out.append({
            'fighter_url': link['href'],
            'first_name': cols[0].text.strip(),
            'last_name': cols[1].text.strip(),
            'nickname': cols[2].text.strip(),
            'height': cols[3].text.strip(),
            'weight': cols[4].text.strip(),
            'reach': cols[5].text.strip(),
            'stance': cols[6].text.strip(),
            'wins': cols[7].text.strip(),
            'losses': cols[8].text.strip(),
            'draws': cols[9].text.strip(),
        })
    return out

all_fighters = []
print('Scraping fighter directory...')
with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = {ex.submit(scrape_fighters_for_letter, c): c
            for c in 'abcdefghijklmnopqrstuvwxyz'}
    for fut in tqdm(concurrent.futures.as_completed(futs),
                    total=len(futs), desc='Letters'):
        res = fut.result()
        if res:
            all_fighters.extend(res)

fighters_df = pd.DataFrame(all_fighters)
fighters_df.to_csv(f'{DATA_DIR}/fighters.csv', index=False)
print(f'\nSaved {len(fighters_df)} fighters to data/fighters.csv')
fighters_df.head()

Scraping fighter directory...


Letters:   0%|          | 0/26 [00:00<?, ?it/s]

Letters: 100%|██████████| 26/26 [00:07<00:00,  3.46it/s]


Saved 4486 fighters to data/fighters.csv


,fighter_url,first_name,last_name,nickname,height,weight,reach,stance,wins,losses,draws
0,http://www.ufcstats.com/fighter-details/93fe73...,Tom,Aaron,,--,155 lbs.,--,,5,3,0
1,http://www.ufcstats.com/fighter-details/15df64...,Danny,Abbadi,The Assassin,"5' 11""",155 lbs.,--,Orthodox,4,6,0
2,http://www.ufcstats.com/fighter-details/59a9d6...,Nariman,Abbasov,Bayraktar,"5' 8""",155 lbs.,"66.0""",Orthodox,28,4,0
3,http://www.ufcstats.com/fighter-details/496146...,Darion,Abbey,,"6' 2""",265 lbs.,"80.0""",Orthodox,9,5,0
4,http://www.ufcstats.com/fighter-details/b36118...,David,Abbott,Tank,"6' 0""",265 lbs.,--,Switch,10,15,0


In [5]:
# notebooks/01_scraper.ipynb — Cell 5: Scrape Fighter Profiles
# THIS WAS MISSING BEFORE — follows each fighter_url to get:
# SLpM, SApM, Str.Acc, Str.Def, TD Avg, TD Acc, TD Def, Sub Avg, DOB
# Output: data/fighters_full.csv

def scrape_fighter_profile(url):
    if not url or pd.isna(url):
        return None
    try:
        soup = get_soup(url)
        stats = {'fighter_url': url}
        key_map = {
            'SLpM': 'slpm',
            'Str. Acc.': 'str_acc_pct',
            'SApM': 'sapm',
            'Str. Def': 'str_def_pct',
            'Str. Def.': 'str_def_pct',
            'TD Avg.': 'td_avg',
            'TD Acc.': 'td_acc_pct',
            'TD Def.': 'td_def_pct',
            'Sub. Avg.': 'sub_avg',
        }
        for li in soup.select('li.b-list__box-list-item'):
            text = li.get_text(separator='|', strip=True)
            if ':' not in text:
                continue
            key, val = text.split(':', 1)
            key, val = key.strip(), val.strip()
            if key in key_map:
                stats[key_map[key]] = val
            elif key == 'DOB':
                stats['dob'] = val
        return stats
    except Exception as e:
        return {'fighter_url': url, 'error': str(e)}

urls = fighters_df['fighter_url'].dropna().unique()
results = []
print(f'Scraping {len(urls)} fighter profiles...')
print('Getting SLpM, SApM, Str.Acc, Str.Def, TD Avg, TD Acc, TD Def, Sub Avg, DOB')

with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = {ex.submit(scrape_fighter_profile, u): u for u in urls}
    for fut in tqdm(concurrent.futures.as_completed(futs),
                    total=len(futs), desc='Profiles'):
        res = fut.result()
        if res:
            results.append(res)

profiles_df = pd.DataFrame(results)
if 'error' in profiles_df.columns:
    errs = profiles_df['error'].notna().sum()
    print(f'Errors: {errs}')
    profiles_df = profiles_df[profiles_df['error'].isna()].drop(columns=['error'])

fighters_full_df = fighters_df.merge(profiles_df, on='fighter_url', how='left')
fighters_full_df.to_csv(f'{DATA_DIR}/fighters_full.csv', index=False)

print(f'\nSaved {len(fighters_full_df)} fighters to data/fighters_full.csv')
print(f'\nProfile stat coverage:')
for col in ['slpm','sapm','str_acc_pct','str_def_pct',
            'td_avg','td_acc_pct','td_def_pct','sub_avg','dob']:
    if col in fighters_full_df.columns:
        nn = fighters_full_df[col].notna().sum()
        print(f'  {col:15s} {nn:>5d} / {len(fighters_full_df)}')
    else:
        print(f'  {col:15s} MISSING')
fighters_full_df.head()

Scraping 4486 fighter profiles...
Getting SLpM, SApM, Str.Acc, Str.Def, TD Avg, TD Acc, TD Def, Sub Avg, DOB


Profiles: 100%|██████████| 4486/4486 [04:25<00:00, 16.89it/s]


Saved 4486 fighters to data/fighters_full.csv

Profile stat coverage:
  slpm             4486 / 4486
  sapm             4486 / 4486
  str_acc_pct      4486 / 4486
  str_def_pct      4486 / 4486
  td_avg           4486 / 4486
  td_acc_pct       4486 / 4486
  td_def_pct       4486 / 4486
  sub_avg          4486 / 4486
  dob              4486 / 4486


,fighter_url,first_name,last_name,nickname,height,weight,reach,stance,wins,losses,draws,dob,slpm,str_acc_pct,sapm,str_def_pct,td_avg,td_acc_pct,td_def_pct,sub_avg
0,http://www.ufcstats.com/fighter-details/93fe73...,Tom,Aaron,,--,155 lbs.,--,,5,3,0,"|Jul 13, 1978",|0.00,|0%,|0.00,|0%,|0.00,|0%,|0%,|0.0
1,http://www.ufcstats.com/fighter-details/15df64...,Danny,Abbadi,The Assassin,"5' 11""",155 lbs.,--,Orthodox,4,6,0,"|Jul 03, 1983",|3.29,|38%,|4.41,|57%,|0.00,|0%,|77%,|0.0
2,http://www.ufcstats.com/fighter-details/59a9d6...,Nariman,Abbasov,Bayraktar,"5' 8""",155 lbs.,"66.0""",Orthodox,28,4,0,"|Feb 01, 1994",|3.00,|20%,|5.67,|46%,|0.00,|0%,|66%,|0.0
3,http://www.ufcstats.com/fighter-details/496146...,Darion,Abbey,,"6' 2""",265 lbs.,"80.0""",Orthodox,9,5,0,"|Feb 25, 1993",|8.44,|50%,|14.06,|28%,|0.00,|0%,|0%,|0.0
4,http://www.ufcstats.com/fighter-details/b36118...,David,Abbott,Tank,"6' 0""",265 lbs.,--,Switch,10,15,0,"|Apr 26, 1965",|1.35,|30%,|3.55,|38%,|1.07,|33%,|66%,|0.0


In [6]:
# notebooks/01_scraper.ipynb — Cell 6: Scrape Fight Details
# Gives TRUE red/blue corner order + W/L status + full X-of-Y stats
# Output: data/fight_details.json

def parse_fight_table(table):
    headers = [th.text.strip() for th in table.select('thead th')]
    rows_data = []
    for row in table.select('tbody tr'):
        cols = row.select('td')
        row_vals = []
        for col in cols:
            ps = col.select('p')
            if ps:
                row_vals.append([p.text.strip() for p in ps])
            else:
                row_vals.append([col.text.strip()])
        rows_data.append(row_vals)
    return {'headers': headers, 'rows': rows_data}

def scrape_fight_details(url):
    if not url:
        return None
    try:
        soup = get_soup(url)
        d = {'fight_url': url}
        # Fighter names + W/L (red corner first)
        persons = soup.select_one('div.b-fight-details__persons')
        if persons:
            for i, fp in enumerate(persons.select('div.b-fight-details__person'), 1):
                name = fp.select_one('a.b-fight-details__person-link')
                status = fp.select_one('i.b-fight-details__person-status')
                d[f'fighter_{i}_name'] = name.text.strip() if name else ''
                d[f'fighter_{i}_status'] = status.text.strip() if status else ''
        # Method, round, time
        for p in soup.select('div.b-fight-details__content p.b-fight-details__text'):
            text = p.get_text(separator='|', strip=True)
            for item in text.split('|'):
                if ':' in item:
                    k, v = item.split(':', 1)
                    d[k.strip().lower().replace(' ', '_')] = v.strip()
        # Stats tables
        tables = soup.select('table.b-fight-details__table')
        if len(tables) >= 1:
            d['totals'] = parse_fight_table(tables[0])
        if len(tables) >= 2:
            d['significant_strikes'] = parse_fight_table(tables[1])
        return d
    except:
        return None

fight_urls = fights_raw_df['fight_url'].dropna().unique()
fight_details_list = []
CHECKPOINT = 500

print(f'Scraping {len(fight_urls)} fight detail pages...')
with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = {ex.submit(scrape_fight_details, u): u for u in fight_urls}
    for fut in tqdm(concurrent.futures.as_completed(futs),
                    total=len(futs), desc='Fight details'):
        res = fut.result()
        if res:
            fight_details_list.append(res)
        if len(fight_details_list) % CHECKPOINT == 0 and fight_details_list:
            with open(f'{DATA_DIR}/fight_details_checkpoint.json', 'w') as f:
                json.dump(fight_details_list, f)

with open(f'{DATA_DIR}/fight_details.json', 'w') as f:
    json.dump(fight_details_list, f)
print(f'\nSaved {len(fight_details_list)} records to data/fight_details.json')

Scraping 8637 fight detail pages...


Fight details: 100%|██████████| 8637/8637 [12:29<00:00, 11.52it/s]



Saved 8637 records to data/fight_details.json


In [7]:
# notebooks/01_scraper.ipynb — Cell 7: Rebuild fights.csv
# Event page: winner first. Detail page: red corner first.
# Use detail page for corner assignment, event page for stat mapping.
# Draw/NC detection: direct comparison, NOT regex.
# Output: data/fights.csv

detail_lookup = {}
for fd in fight_details_list:
    url = fd.get('fight_url', '')
    f1 = fd.get('fighter_1_name', '').strip()
    f2 = fd.get('fighter_2_name', '').strip()
    s1 = fd.get('fighter_1_status', '').strip()
    s2 = fd.get('fighter_2_status', '').strip()
    if url and f1:
        detail_lookup[url] = {
            'red': f1, 'blue': f2, 'rs': s1, 'bs': s2
        }

rebuilt = []
skipped = 0
for _, row in fights_raw_df.iterrows():
    url = row['fight_url']
    if url not in detail_lookup:
        skipped += 1
        continue
    dl = detail_lookup[url]
    red, blue = dl['red'], dl['blue']
    winner = red if dl['rs']=='W' else (blue if dl['bs']=='W' else 'Draw/NC')
    ew = row['fighter_1_winner']  # event page winner
    if red == ew:
        rk,bk = row['winner_kd'],row['loser_kd']
        rs,bs = row['winner_str'],row['loser_str']
        rt,bt = row['winner_td'],row['loser_td']
        ru,bu = row['winner_sub'],row['loser_sub']
    else:
        rk,bk = row['loser_kd'],row['winner_kd']
        rs,bs = row['loser_str'],row['winner_str']
        rt,bt = row['loser_td'],row['winner_td']
        ru,bu = row['loser_sub'],row['winner_sub']
    rebuilt.append({
        'event_name': row['event_name'], 'event_date': row['event_date'],
        'fight_url': url, 'fighter_1': red, 'fighter_2': blue,
        'winner': winner,
        'f1_kd': rk, 'f2_kd': bk, 'f1_str': rs, 'f2_str': bs,
        'f1_td': rt, 'f2_td': bt, 'f1_sub': ru, 'f2_sub': bu,
        'weight_class': row['weight_class'], 'method': row['method'],
        'round': row['round'], 'time': row['time'],
    })

fights_df = pd.DataFrame(rebuilt)
decided = (fights_df['winner']==fights_df['fighter_1']) | \
          (fights_df['winner']==fights_df['fighter_2'])
f1w = (fights_df['winner']==fights_df['fighter_1']).sum()
f2w = (fights_df['winner']==fights_df['fighter_2']).sum()
dnc = (~decided).sum()

fights_df.to_csv(f'{DATA_DIR}/fights.csv', index=False)
print(f'Saved {len(fights_df)} fights to data/fights.csv')
print(f'Skipped (no detail): {skipped}')
print(f'Red wins:  {f1w} ({f1w/len(fights_df):.1%})')
print(f'Blue wins: {f2w} ({f2w/len(fights_df):.1%})')
print(f'Draw/NC:   {dnc} ({dnc/len(fights_df):.1%})')
print(f'Red WR (decided): {f1w/(f1w+f2w):.1%}')
fights_df.head()

Saved 8637 fights to data/fights.csv
Skipped (no detail): 0
Red wins:  5454 (63.1%)
Blue wins: 3029 (35.1%)
Draw/NC:   154 (1.8%)
Red WR (decided): 64.3%


,event_name,event_date,fight_url,fighter_1,fighter_2,winner,f1_kd,f2_kd,f1_str,f2_str,f1_td,f2_td,f1_sub,f2_sub,weight_class,method,round,time
0,UFC 327: Prochazka vs. Ulberg,"April 11, 2026",http://www.ufcstats.com/fight-details/a031f062...,Jiri Prochazka,Carlos Ulberg,Carlos Ulberg,0,1,14,27,0,0,0,0,Light Heavyweight,KO/TKO\n\n \n\n Punch,1,3:45
1,UFC 327: Prochazka vs. Ulberg,"April 11, 2026",http://www.ufcstats.com/fight-details/aade9d76...,Azamat Murzakanov,Paulo Costa,Paulo Costa,0,1,34,55,1,0,1,0,Light Heavyweight,KO/TKO\n\n \n\n Kick,3,1:23
2,UFC 327: Prochazka vs. Ulberg,"April 11, 2026",http://www.ufcstats.com/fight-details/903552ed...,Curtis Blaydes,Josh Hokit,Josh Hokit,0,0,174,177,2,0,0,0,Heavyweight,U-DEC,3,5:00
3,UFC 327: Prochazka vs. Ulberg,"April 11, 2026",http://www.ufcstats.com/fight-details/c4d060cf...,Dominick Reyes,Johnny Walker,Dominick Reyes,0,0,34,42,0,0,0,0,Light Heavyweight,S-DEC,3,5:00
4,UFC 327: Prochazka vs. Ulberg,"April 11, 2026",http://www.ufcstats.com/fight-details/60949f33...,Cub Swanson,Nate Landwehr,Cub Swanson,2,0,37,15,0,0,0,0,Featherweight,KO/TKO\n\n \n\n Punch,1,4:06


In [8]:
# notebooks/01_scraper.ipynb — Cell 8: Summary

print('=' * 60)
print('SCRAPING COMPLETE')
print('=' * 60)

files = {
    'events.csv': len(events_df),
    'fights_raw.csv': len(fights_raw_df),
    'fights.csv': len(fights_df),
    'fighters.csv': len(fighters_df),
    'fighters_full.csv': len(fighters_full_df),
    'fight_details.json': len(fight_details_list),
}
for fname, count in files.items():
    path = f'{DATA_DIR}/{fname}'
    size = os.path.getsize(path) / 1024 if os.path.exists(path) else 0
    print(f'  {fname:30s} {count:>6d} rows  {size:>8.1f} KB')

print(f'\nFighter profile stats:')
for col in ['slpm','sapm','str_acc_pct','str_def_pct',
            'td_avg','td_acc_pct','td_def_pct','sub_avg','dob']:
    if col in fighters_full_df.columns:
        nn = fighters_full_df[col].notna().sum()
        print(f'  {col:15s} {nn:>5d} / {len(fighters_full_df)}')

print(f'\nNext: open 02_data_cleaning.ipynb')

SCRAPING COMPLETE
  events.csv                        769 rows     100.3 KB
  fights_raw.csv                   8637 rows    1632.3 KB
  fights.csv                       8637 rows    1751.5 KB
  fighters.csv                     4486 rows     506.2 KB
  fighters_full.csv                4486 rows     754.5 KB
  fight_details.json               8637 rows   13470.1 KB

Fighter profile stats:
  slpm             4486 / 4486
  sapm             4486 / 4486
  str_acc_pct      4486 / 4486
  str_def_pct      4486 / 4486
  td_avg           4486 / 4486
  td_acc_pct       4486 / 4486
  td_def_pct       4486 / 4486
  sub_avg          4486 / 4486
  dob              4486 / 4486

Next: open 02_data_cleaning.ipynb
